# DiD-BCF — D_contamination (linearity_degree=1)

**Workstream D · staggered adoption (cohort x event-time effects)**

stronger dynamics -> larger TWFE contamination

Fits DiD-BCF on the `D_contamination` scenario at `linearity_degree=1` and reports
metrics for **both** the plain DiD-BCF posterior and the proposed **posterior
correction** (Algorithm 1 of the theory note), so the correction can be judged
directly. Panel: N=200, 4 pre + 4 post periods.

> **Colab:** upload just this notebook and *Run all* — the first cell installs the
> dependencies and the second clones the engine automatically.

In [1]:
# Colab: install the DiD-BCF dependencies (stochtree provides the BCF sampler).
%pip install -q stochtree scikit-learn joblib tqdm pandas numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 36.9 MB/s eta 0:00:00


In [2]:
import os, sys

# --- Locate the DiD-BCF engine ------------------------------------------------
# So you can upload just THIS notebook to Colab and Run all. Resolution order:
#   1. `did_bcf_revision` already importable;
#   2. running inside a repo checkout (the parent folder holds the package);
#   3. otherwise clone https://github.com/hugogobato/DiD-BCF and use it.
REPO_URL = "https://github.com/hugogobato/DiD-BCF.git"
ENGINE_SUBDIR = os.path.join("DiD-BCF", "Simulation_Studies_Revision")

def _locate_root():
    try:
        import did_bcf_revision  # noqa: F401
        return os.path.dirname(os.path.dirname(did_bcf_revision.__file__))
    except Exception:
        pass
    parent = os.path.abspath(os.path.join(os.getcwd(), ".."))
    if os.path.isdir(os.path.join(parent, "did_bcf_revision")):
        return parent
    if not os.path.isdir("DiD-BCF"):
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL], check=True)
    return os.path.abspath(ENGINE_SUBDIR)

ROOT = _locate_root()
sys.path.insert(0, ROOT)
print("Using DiD-BCF engine at:", ROOT)

from did_bcf_revision.runner import run_named
from did_bcf_revision.metrics import (compute_metrics, plain_vs_corrected,
                                      surface_metrics)

Using DiD-BCF engine at: /content/DiD-BCF/Simulation_Studies_Revision


In [3]:
REPS = 100      # replications (lower for a quick smoke test)
JOBS = 1        # parallel reps (keep 1 on a single-core/GPU Colab)

bcf_params = dict(num_gfr=50, num_mcmc=500, keep_every=5, num_chains=3)

summaries = run_named(
    "D_contamination",
    linearity_degree=1,
    reps=REPS,
    jobs=JOBS,
    bcf_params=bcf_params,
    prop_method="logit",   # pilot propensity for the posterior correction
    n_splits=2,            # cross-fitting folds for the correction
)
summaries.head()

[D_contamination_lin_1] staggered DGP | N=(200,) | reps=100 | 100 fits | jobs=1


D_contamination: 100%|██████████| 100/100 [6:08:51<00:00, 221.32s/fit]

[D_contamination_lin_1] wrote 3000 rows -> /content/DiD-BCF/Simulation_Studies_Revision/Results/summaries_D_contamination_lin_1.csv


,dgp,setting,linearity_degree,N,rep,estimand_type,estimand_id,g,t,k,...,p_bayes,surf_rmse,surf_mae,surf_n,surf_mape,surf_cover95,surf_len95,surf_cover90,surf_len90,true
0,staggered,D_contamination,1,200,0,GATT,g=4_t=4,4.0,4.0,0.0,...,0.284667,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.732024
1,staggered,D_contamination,1,200,0,GATT,g=4_t=5,4.0,5.0,1.0,...,0.151333,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.917643
2,staggered,D_contamination,1,200,0,GATT,g=4_t=6,4.0,6.0,2.0,...,0.050667,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7.103263
3,staggered,D_contamination,1,200,0,GATT,g=4_t=7,4.0,7.0,3.0,...,0.012667,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,9.288882
4,staggered,D_contamination,1,200,0,GATT,g=5_t=5,5.0,5.0,0.0,...,0.058667,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.529720


In [4]:
# Decomposed metrics: bias, MC SD/variance, RMSE, MAE, MAPE, coverage 90/95,
# interval length, calibration ratio (avg_post_sd/emp_sd), size/power and their
# Monte-Carlo SEs -- for plain AND corrected DiD-BCF.
metrics = compute_metrics(summaries)
plain_vs_corrected(metrics)

,dgp,setting,linearity_degree,N,estimand_type,estimand_id,role,bias_corrected,bias_plain,cover90_corrected,...,len95_corrected,len95_plain,mae_corrected,mae_plain,reject05_corrected,reject05_plain,rmse_corrected,rmse_plain,sd_ratio_corrected,sd_ratio_plain
0,staggered,D_contamination,1,200,ATT,ATT,power,-2.926275,-6.468118,0.00,...,0.882875,2.676064,2.926275,6.468118,1.0,0.95,2.971803,6.486096,0.402539,1.558834
1,staggered,D_contamination,1,200,ES,k=0,power,-1.048869,-4.699340,0.03,...,0.832336,1.386452,1.051533,4.699340,1.0,0.44,1.113367,4.709037,0.527600,1.882225
2,staggered,D_contamination,1,200,ES,k=1,power,-3.879820,-7.703640,0.00,...,0.999888,2.701205,3.879820,7.703640,1.0,0.97,3.926272,7.726101,0.430630,1.562301
3,staggered,D_contamination,1,200,ES,k=2,power,-4.206393,-7.582942,0.00,...,1.258142,3.893720,4.206393,7.582942,1.0,0.94,4.258442,7.612632,0.420329,1.480462
4,staggered,D_contamination,1,200,ES,k=3,power,-3.050110,-5.811755,0.00,...,1.669251,4.871111,3.050110,5.811755,1.0,0.97,3.150840,5.851569,0.471171,1.693733
5,staggered,D_contamination,1,200,GATT,g=4_t=4,power,-0.017062,-2.748279,0.82,...,1.427996,2.174597,0.334438,2.748279,1.0,0.00,0.416699,2.750937,0.832079,31.073336
6,staggered,D_contamination,1,200,GATT,g=4_t=5,power,-1.217100,-4.027941,0.13,...,1.464929,2.987652,1.219568,4.027941,1.0,0.09,1.327383,4.050207,0.643053,1.791653
7,staggered,D_contamination,1,200,GATT,g=4_t=6,power,-2.263005,-5.038869,0.01,...,1.540274,4.112443,2.263005,5.038869,1.0,0.46,2.363907,5.073677,0.525733,1.734167
8,staggered,D_contamination,1,200,GATT,g=4_t=7,power,-3.050110,-5.811755,0.00,...,1.669251,4.871111,3.050110,5.811755,1.0,0.97,3.150840,5.851569,0.471171,1.693733
9,staggered,D_contamination,1,200,GATT,g=5_t=5,power,-0.664671,-4.582957,0.53,...,1.677967,2.131088,0.684552,4.582957,1.0,0.43,0.809322,4.605353,0.860046,1.363560


## CATT-surface metrics (the paper's headline RMSE/MAE/MAPE)

Within-replication RMSE/MAE/MAPE over the *individual* treated observations
(mean +/- SD across runs) plus the *pointwise* CATT coverage -- the evidence
that DiD-BCF recovers the heterogeneous effect that GATT-only methods cannot.

In [5]:
surface_metrics(summaries)

,dgp,setting,linearity_degree,N,method,n_reps,avg_n_treated_obs,surf_rmse_mean,surf_rmse_sd,surf_mae_mean,...,surf_mape_mean,surf_mape_sd,surf_cover90_mean,surf_cover90_sd,surf_cover95_mean,surf_cover95_sd,surf_len90_mean,surf_len90_sd,surf_len95_mean,surf_len95_sd
0,staggered,D_contamination,1,200,corrected,100,431.6,4.764584,0.410724,3.408725,...,0.355092,0.024863,0.181019,0.027059,0.217142,0.031827,1.421739,0.127959,1.718794,0.162219
1,staggered,D_contamination,1,200,plain,100,431.6,7.504538,0.516294,6.468118,...,0.780487,0.053179,0.012525,0.015941,0.023659,0.020726,3.233731,0.286962,3.864274,0.325804


## Goodman-Bacon decomposition (TWFE contamination)

How much of a naive TWFE estimate on this DGP comes from the
"already-treated as control" comparisons that bias it.

In [6]:
from did_bcf_revision.dgps import generate_staggered_did
from did_bcf_revision.goodman_bacon import bacon_summary

df0 = generate_staggered_did(seed=0, linearity_degree=1)
bacon_summary(df0)

{'twfe': 5.063198924628939,
 'bacon_total': 5.0631989246289395,
 'weight_treated_vs_untreated': 0.6334173693819004,
 'weight_earlier_vs_later': 0.2391665942217972,
 'weight_already_treated': 0.1274160363963024,
 'components':                    type  treat_group  control_group    weight        dd  \
 0      Earlier_vs_Later          4.0            5.0  0.060205  2.641511   
 1      Earlier_vs_Later          4.0            6.0  0.106778  3.036811   
 2      Earlier_vs_Later          5.0            6.0  0.072184  4.176324   
 3      Later_vs_Earlier          5.0            4.0  0.045153  3.585888   
 4      Later_vs_Earlier          6.0            4.0  0.053389  4.402213   
 5      Later_vs_Earlier          6.0            5.0  0.028874  4.134568   
 6  Treated_vs_Untreated          4.0            inf  0.231731  4.719428   
 7  Treated_vs_Untreated          5.0            inf  0.234982  6.264973   
 8  Treated_vs_Untreated          6.0            inf  0.166704  7.176302   
 
    contribut